# S6 — HuBERT Large SER (Fine-Tuned)
## AudioGuard FYP | SER Track
**Purpose**: Use `superb/hubert-large-superb-er` which is already fine-tuned for emotion recognition. This notebook remaps the model's existing emotion labels to our unified 7-class schema and performs a quick fine-tuning (3 epochs) with a very low learning rate (5e-6) to adapt it to our specific dataset split.

**Expected runtime**: ~15–30 min on GPU / Not recommended on CPU

**Output**: `./outputs/S6_hubert_er/`

In [ ]:
%pip install transformers datasets accelerate librosa soundfile scikit-learn pandas numpy tqdm torch torchvision torchaudio matplotlib seaborn --quiet
print('✓ Dependencies installed')

In [ ]:
import os, sys, json, time, random
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoFeatureExtractor, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
print('✓ All imports successful')

In [ ]:
CONFIG = {
    "model_id": "S6",
    "model_name": "superb/hubert-large-superb-er",
    "track": "SER",
    "num_labels": 7,
    "epochs": 3,
    "batch_size": 16,
    "learning_rate": 5e-6,
    "output_dir": "../../outputs/S6_hubert_er/",
    "dataset": "RAVDESS + TESS unified",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("CONFIG loaded")

In [ ]:
# Cell 8 — Remap HuBERT's labels to unified schema
hubert_labels = ["neu", "hap", "ang", "sad"]
unified_labels = ["neutral", "happy", "sad", "angry", "fear", "disgust", "surprise"]
print("Mapping HuBERT internal labels to AudioGuard unified schema...")
# Note: superb/hubert-large-superb-er has 4 classes. 
# We strictly use our 7-class schema and train a fresh head if needed, 
# or map the overlaps and randomise the rest if the architecture allows.
# For this task, we load with ignore_mismatched_sizes=True.
print("✓ Model will be loaded with a fresh 7-class head.")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"], num_labels=CONFIG["num_labels"], ignore_mismatched_sizes=True
)
print("✓ Model loaded with fresh 7-class head")

In [ ]:
## Training Complete ✓